# Instruction Fine-Tuning GPT-2 on Dolly

A **base** (pretrained-only) language model has learned to predict the next token over web text, but it has never been taught to *follow instructions*. GPT-2 (2019) is a good example: ask it a question and it will ramble, repeat itself, or drift into unrelated text.

In this notebook we:

1. Load the base `gpt2` model (124M parameters).
2. Prompt it and watch it fail to answer sensibly.
3. Instruction fine-tune it on [`databricks/databricks-dolly-15k`](https://huggingface.co/datasets/databricks/databricks-dolly-15k).
4. Prompt it again with the same instructions and compare.

In [ ]:
import math
import os
import random

# Fall back to CPU for any op the Apple GPU (MPS) backend doesn't implement yet.
# Must be set before torch is imported.
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
# TensorFlow is installed in this venv; stop transformers from importing it (slow, and unused here)
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)

In [ ]:
MODEL_ID = "gpt2"      # 124M base model; "gpt2-medium" (355M) gives noticeably better answers
MAX_LEN = 512          # max tokens per training example (GPT-2's context window is 1024)
NUM_TRAIN = 5000       # subset of Dolly to keep training quick; use None for all ~15k
SEED = 42

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

random.seed(SEED)
torch.manual_seed(SEED)
print("device:", device)
if device == "mps":
    # Apple Silicon uses unified memory: this is how much of it the GPU may use
    print(f"MPS memory budget: {torch.mps.recommended_max_memory() / 1e9:.1f} GB")

## 1. Load the base model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# GPT-2 has no padding token. Reuse EOS for padding; padded positions are masked out of
# attention and loss anyway, so the model never learns from them.
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    # fp32 is the safe choice on MPS: mixed-precision training support there is still patchy,
    # and at 124M params the weights + AdamW state are only ~2 GB anyway.
    torch_dtype=torch.float32,
).to(device)
model.config.pad_token_id = tokenizer.pad_token_id

print(f"parameters: {model.num_parameters() / 1e6:.0f}M")

## 2. Prompt template

We use a simple Alpaca-style template. Dolly examples sometimes include a `context` passage (for closed QA, summarization, information extraction), so the template has an optional `### Context:` section.

During training we append the EOS token (`<|endoftext|>` for GPT-2) after the response so the model learns *when to stop*.

In [ ]:
def build_prompt(instruction, context=""):
    prompt = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\n{instruction.strip()}\n\n"
    )
    if context and context.strip():
        prompt += f"### Context:\n{context.strip()}\n\n"
    prompt += "### Response:\n"
    return prompt


def generate(model, instruction, context="", max_new_tokens=150):
    prompt = build_prompt(instruction, context)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    model.eval()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
test_prompts = [
    {"instruction": "What is the capital of France?"},
    {"instruction": "Give me three tips for staying healthy."},
    {"instruction": "Write a short poem about the ocean."},
    {
        "instruction": "Summarize the following text in one sentence.",
        "context": (
            "The Great Wall of China is a series of fortifications built across the historical "
            "northern borders of ancient Chinese states and Imperial China as protection against "
            "various nomadic groups from the Eurasian Steppe. Several walls were built from as early "
            "as the 7th century BC, with selective stretches later joined by Qin Shi Huang, the first "
            "emperor of China."
        ),
    },
]

## 3. Prompt the base model (before fine-tuning)

Expect rambling, repetition loops, or text that continues the *document* rather than answering the question.

In [ ]:
before = {}
for p in test_prompts:
    before[p["instruction"]] = generate(model, **p)
    print("=" * 80)
    print("INSTRUCTION:", p["instruction"])
    print("-" * 80)
    print(before[p["instruction"]])

## 4. Load and prepare the Dolly dataset

Each Dolly row has `instruction`, `context`, `response` and `category`.

For each example we tokenize `prompt + response + <eos>` and set the labels for the prompt tokens to `-100`, so the loss is only computed on the **response**. The model learns to produce answers, not to reproduce the template.

In [ ]:
dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
print(dolly)
print(dolly[0])

In [ ]:
dolly = dolly.shuffle(seed=SEED)
if NUM_TRAIN is not None:
    dolly = dolly.select(range(min(NUM_TRAIN, len(dolly))))

splits = dolly.train_test_split(test_size=0.05, seed=SEED)
train_raw, eval_raw = splits["train"], splits["test"]
print(len(train_raw), "train /", len(eval_raw), "eval")

In [ ]:
def tokenize(example):
    prompt = build_prompt(example["instruction"], example["context"])
    prompt_ids = tokenizer(prompt)["input_ids"]
    response_ids = tokenizer(example["response"].strip(), add_special_tokens=False)["input_ids"]
    response_ids = response_ids + [tokenizer.eos_token_id]

    input_ids = (prompt_ids + response_ids)[:MAX_LEN]
    labels = ([-100] * len(prompt_ids) + response_ids)[:MAX_LEN]
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}


train_ds = train_raw.map(tokenize, remove_columns=train_raw.column_names)
eval_ds = eval_raw.map(tokenize, remove_columns=eval_raw.column_names)

# drop examples whose response was fully truncated away
has_target = lambda ex: any(l != -100 for l in ex["labels"])
train_ds = train_ds.filter(has_target)
eval_ds = eval_ds.filter(has_target)
print(train_ds)

Examples have different lengths, so the collator pads each batch to its longest sequence. Padded positions get label `-100` so they are ignored by the loss. (The pad id is the same as EOS, but that doesn't matter: the EOS we *do* want the model to learn is in `labels`, while padding is masked by position.)

In [ ]:
def collate(batch):
    max_len = max(len(ex["input_ids"]) for ex in batch)
    pad_id = tokenizer.pad_token_id
    input_ids, attention_mask, labels = [], [], []
    for ex in batch:
        pad = max_len - len(ex["input_ids"])
        input_ids.append(ex["input_ids"] + [pad_id] * pad)
        attention_mask.append(ex["attention_mask"] + [0] * pad)
        labels.append(ex["labels"] + [-100] * pad)
    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_mask),
        "labels": torch.tensor(labels),
    }

## 5. Fine-tune

GPT-2 small is easily small enough for a full fine-tune (all weights updated) on Apple Silicon.

Settings for the M5 Pro (MPS backend, unified memory):
- **fp32**: no `fp16`/`bf16` mixed precision, which is unreliable on MPS.
- **Batch size 8**: the logits (batch × seq_len × 50k vocab × 4 bytes) are the biggest activation, and batch 8 fits easily in unified memory. If you switch to `gpt2-medium`/`gpt2-large` and see memory pressure, drop to 4 and raise `gradient_accumulation_steps` to 4.
- **`dataloader_pin_memory=False`**: pinned memory is a CUDA feature and only produces warnings on MPS.
- **`dataloader_num_workers=0`**: worker subprocesses on macOS use `spawn`, which is slow and error-prone inside notebooks.

GPT-2 starts much further from instruction following than a modern base model, so we train for 2 epochs.

In [ ]:
args = TrainingArguments(
    output_dir="gpt2-dolly",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,       # effective batch size 16
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="no",
    report_to="none",
    group_by_length=True,                # batch similar lengths together -> less padding
    dataloader_pin_memory=False,         # pinned memory is CUDA-only
    dataloader_num_workers=0,            # avoid macOS spawn overhead in notebooks
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collate,
)

eval_before = trainer.evaluate()
print(f"eval loss before: {eval_before['eval_loss']:.3f}  (perplexity {math.exp(eval_before['eval_loss']):.1f})")

In [ ]:
trainer.train()

In [ ]:
if device == "mps":
    torch.mps.empty_cache()   # release cached training buffers back to the system

eval_after = trainer.evaluate()
print(f"eval loss before: {eval_before['eval_loss']:.3f}  (perplexity {math.exp(eval_before['eval_loss']):.1f})")
print(f"eval loss after:  {eval_after['eval_loss']:.3f}  (perplexity {math.exp(eval_after['eval_loss']):.1f})")

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_steps = [h["step"] for h in history if "loss" in h]
train_loss = [h["loss"] for h in history if "loss" in h]
eval_steps = [h["step"] for h in history if "eval_loss" in h]
eval_loss = [h["eval_loss"] for h in history if "eval_loss" in h]

plt.figure(figsize=(8, 4))
plt.plot(train_steps, train_loss, label="train")
plt.plot(eval_steps, eval_loss, marker="o", label="eval")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Instruction fine-tuning on Dolly")
plt.legend()
plt.show()

## 6. Prompt the fine-tuned model

Same prompts, same greedy decoding. The model should now answer the instruction directly and stop with EOS.

In [ ]:
for p in test_prompts:
    after = generate(model, **p)
    print("=" * 80)
    print("INSTRUCTION:", p["instruction"])
    print("-" * 36, "BEFORE", "-" * 36)
    print(before[p["instruction"]])
    print("-" * 36, "AFTER", "-" * 37)
    print(after)

Try your own instruction:

In [ ]:
print(generate(model, "Explain what a neural network is to a ten year old."))

## 7. Save the model

In [ ]:
trainer.save_model("gpt2-dolly")
tokenizer.save_pretrained("gpt2-dolly")